# otto-GPT(318M) 멀티태스크 instruction 튜닝

**318M 사전학습 베이스**에 기능 학습: 대화/QA(KoAlpaca) + 도구 호출(calculator/weather/search/**rag**).
- 통일 포맷 + 프롬프트 마스킹(응답만 학습) / 도구 ×3 / 지식→rag
- 출력: `otto_gpt_350m_instruct.pt` (57M 것과 별도)

선행: `otto_gpt_scratch.ipynb` 로 318M 베이스(`otto_gpt_350m_best.pt`) 학습 완료.

In [ ]:
!pip -q install sentencepiece datasets
import os, math, time, random, numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR='/content/drive/MyDrive/otto_gpt'
SPM_PREFIX=f'{PROJECT_DIR}/tokenizer_ko'
# 318M 베이스 우선 (best→latest), 없으면 57M 폴백
for _cand in ['otto_gpt_350m_best.pt','otto_gpt_350m.pt','otto_gpt_best.pt','otto_gpt.pt']:
    BASE_CKPT=f'{PROJECT_DIR}/ckpt/{_cand}'
    if os.path.exists(BASE_CKPT): break
INST_CKPT=f'{PROJECT_DIR}/ckpt/otto_gpt_350m_instruct.pt'   # 318M instruct (57M 것과 별도)
assert os.path.exists(BASE_CKPT), '베이스 체크포인트 없음. scratch 노트북 먼저.'
print('베이스:', BASE_CKPT)

In [ ]:
import sentencepiece as spm
sp=spm.SentencePieceProcessor(model_file=SPM_PREFIX+'.model')
print('어휘 크기:', sp.get_piece_size())

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class GPTConfig:
    vocab_size: int = 16000
    block_size: int = 512     
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 624
    dropout: float = 0.1
    bias: bool = False

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.c_attn = nn.Linear(cfg.n_embd, 3*cfg.n_embd, bias=cfg.bias)
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        self.n_head, self.n_embd, self.dropout = cfg.n_head, cfg.n_embd, cfg.dropout

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C//self.n_head).transpose(1, 2)
        
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4*cfg.n_embd, bias=cfg.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4*cfg.n_embd, cfg.n_embd, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd, bias=cfg.bias)
        self.mlp = MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(cfg.vocab_size, cfg.n_embd),   
            wpe=nn.Embedding(cfg.block_size, cfg.n_embd),   
            drop=nn.Dropout(cfg.dropout),
            h=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)]),
            ln_f=nn.LayerNorm(cfg.n_embd, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight   
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2*cfg.n_layer))

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def num_params(self):
        n = sum(p.numel() for p in self.parameters())
        return n - self.transformer.wpe.weight.numel()

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x = self.transformer.drop(self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-1)
            return logits, loss
        logits = self.lm_head(x[:, [-1], :])
        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("GPT 모델 정의 완료")

In [ ]:
device='cuda' if torch.cuda.is_available() else 'cpu'
dtype='bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
ckpt=torch.load(BASE_CKPT, map_location=device)
config=GPTConfig(**ckpt['config'])   # 318M config 자동 로드
model=GPT(config).to(device); model.load_state_dict(ckpt['model'])
block_size=config.block_size
print(f'베이스 로드 완료: {model.num_params()/1e6:.1f}M, 사전학습 step:', ckpt.get('iter_num'))

In [ ]:
# ── 멀티태스크 데이터 (통일 포맷 + 프롬프트 마스킹) ──
from datasets import load_dataset
random.seed(0)
P_NOINPUT='### 지시:\n{instruction}\n\n### 응답:\n'
P_INPUT  ='### 지시:\n{instruction}\n\n### 입력:\n{input}\n\n### 응답:\n'
def format_prompt(ex):
    if ex.get('input'): return P_INPUT.format(instruction=ex['instruction'], input=ex['input'])
    return P_NOINPUT.format(instruction=ex['instruction'])
def encode_masked(prompt, response):
    p=sp.encode(prompt); r=sp.encode(response)+[sp.eos_id()]
    ids=(p+r)[:block_size]; label=([-1]*len(p)+r)[:block_size]
    x=ids[:-1]; y=label[1:]
    x=x+[sp.pad_id()]*(block_size-1-len(x)); y=y+[-1]*(block_size-1-len(y))
    return x,y

examples=[]
for ex in load_dataset('beomi/KoAlpaca-v1.1a', split='train'):
    instr=(ex.get('instruction') or '').strip(); inp=(ex.get('input') or '').strip(); out=str(ex.get('output') or '').strip()
    if len(instr)<2 or len(out)<2: continue
    examples.append({'instruction':instr,'input':inp,'output':out})
n_ka=len(examples)

calc=[]
for _ in range(400):
    a=random.randint(1,999); b=random.randint(1,99); op=random.choice(['+','-','*','/'])
    e=f'{a}{op}{b}'; t=random.choice(['{e} 계산해줘','{e} 얼마야?','{e} 결과는?','{e} 계산'])
    calc.append({'instruction':t.format(e=e),'input':'','output':f'calculator(수식="{e}")'})
cities=['서울','부산','대구','인천','광주','대전','울산','제주','수원','성남','고양','용인','창원','청주','전주','천안','김해','포항','춘천','강릉','목포','여수','속초','경주','안동','부천','평택','진주','군산']
weather=[{'instruction':t.format(c=c),'input':'','output':f'weather(도시="{c}")'}
         for c in cities for t in ['{c} 날씨 어때?','{c} 지금 비 와?','오늘 {c} 기온 알려줘','{c} 주말 날씨']]
queries=['오늘 환율','파이썬 설치법','근처 맛집','지하철 노선','영화 예매','기차 시간표','주식 시세','버스 도착','택배 조회','로또 번호']
search=[{'instruction':t.format(q=q),'input':'','output':f'search(검색어="{q}")'}
        for q in queries for t in ['{q} 검색해줘','{q} 좀 찾아줘','{q} 검색']]
topics=['인공지능','머신러닝','딥러닝','블록체인','광합성','상대성이론','한국의 역사','분산 시스템','양자역학','진화론','민주주의','자본주의','중력','DNA','인터넷','반도체','기후변화','백신','알고리즘','데이터베이스']
rag=[{'instruction':t.format(x=x),'input':'','output':f'rag(질문="{x}")'}
     for x in topics for t in ['{x}이 뭐야?','{x}에 대해 알려줘','{x} 설명해줘','{x}의 정의는?','{x} 개념 알려줘']]
tool_examples=calc+weather+search+rag
examples += tool_examples*3
random.shuffle(examples)
print(f'총 {len(examples):,} = KoAlpaca {n_ka:,} + 도구 {len(tool_examples)}×3')

X=np.zeros((len(examples),block_size-1),dtype=np.int32)
Y=np.full((len(examples),block_size-1),-1,dtype=np.int32)
kept=0
for ex in examples:
    x,y=encode_masked(format_prompt(ex), ex['output'])
    if all(t==-1 for t in y): continue
    X[kept]=x; Y[kept]=y; kept+=1
X=X[:kept]; Y=Y[:kept]
print(f'토크나이징 완료: {kept:,} 샘플 {X.shape}')

In [ ]:
# ── 학습 (응답 토큰만 loss). 318M은 메모리 큼 → 배치 자동 조절 ──
Xt=torch.from_numpy(X).long(); Yt=torch.from_numpy(Y).long()
if torch.cuda.is_available():
    _v=torch.cuda.get_device_properties(0).total_memory/1e9
    batch_size = 8 if _v>=38 else (2 if _v>=22 else 1)   # A100 / L4 / T4
else:
    batch_size = 2
max_iters=4000; lr=1e-4; min_lr=1e-5; warmup=100
print('batch_size=', batch_size)
def get_batch():
    ix=torch.randint(0,Xt.size(0),(batch_size,)); return Xt[ix].to(device), Yt[ix].to(device)
opt=torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
scaler=torch.amp.GradScaler(enabled=(dtype=='float16'))
def get_lr(it):
    if it<warmup: return lr*(it+1)/warmup
    r=(it-warmup)/(max_iters-warmup); return min_lr+0.5*(1+math.cos(math.pi*r))*(lr-min_lr)
model.train(); t0=time.time()
for it in range(max_iters):
    for g in opt.param_groups: g['lr']=get_lr(it)
    X_,Y_=get_batch()
    with torch.amp.autocast(device_type='cuda',dtype=getattr(torch,dtype)) if device=='cuda' else torch.enable_grad():
        _,loss=model(X_,Y_)
    opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
    scaler.step(opt); scaler.update()
    if it%200==0: print(f'iter {it:4d} | loss {loss.item():.4f} | {time.time()-t0:.0f}s')
torch.save({'model':model.state_dict(),'config':config.__dict__,'iter_num':it+1}, INST_CKPT)
print('튜닝 완료, 저장:', INST_CKPT)

In [ ]:
# ── 데모: 318M 라우팅 + 대화 ──
@torch.no_grad()
def run(instruction, inp='', max_new_tokens=120, temperature=0.7, top_k=40, rep_penalty=1.3):
    model.eval()
    x=torch.tensor([sp.encode(format_prompt({'instruction':instruction,'input':inp}))],dtype=torch.long,device=device)
    for _ in range(max_new_tokens):
        with torch.amp.autocast(device_type='cuda',dtype=getattr(torch,dtype)) if device=='cuda' else torch.no_grad():
            logits,_=model(x[:,-block_size:])
        logits=logits[:,-1,:]/temperature
        for t in set(x[0].tolist()): logits[0,t]/=rep_penalty
        v,_=torch.topk(logits, min(top_k,logits.size(-1))); logits[logits<v[:,[-1]]]=-float('inf')
        nxt=torch.multinomial(F.softmax(logits,dim=-1),1)
        if nxt.item()==sp.eos_id(): break
        x=torch.cat((x,nxt),dim=1)
    return sp.decode(x[0].tolist()).split('### 응답:')[-1].strip()

print('[계산]')
for q in ['12*7 계산해줘','345+678 얼마야?']: print('Q:',q,'\n→',run(q),'\n')
print('[지식 → rag]')
for q in ['인공지능이 뭐야?','광합성 설명해줘']: print('Q:',q,'\n→',run(q),'\n')
print('[대화/도구 혼합]')
for q in ['부산 날씨 어때?','건강하게 사는 방법 알려줘']: print('Q:',q,'\n→',run(q),'\n')